# 第315章: Pix2Pix と ControlNet の直感

## 📋 この章で学ぶこと

- [ ] 画像から画像への変換（Image-to-Image Translation）の概念を理解できる
- [ ] Pix2Pix（cGAN）とControlNet（条件付き拡散モデル）のアーキテクチャの違いを説明できる
- [ ] U-Netに「空間的な条件（エッジや輪郭）」を入力するメカニズムを実装できる
- [ ] Zero-Conv（ゼロ畳み込み）が事前学習を破壊せずに条件を追加する仕組みを理解できる

## 🎯 前提知識

- ✅ Notebook 312（テキスト条件付き拡散モデル）

⏱️ **推定学習時間**: 90-120分
📊 **難易度**: ★★★★☆（上級）
🎓 **カテゴリ**: 理論・実践

---

## 🌟 はじめに

これまで、画像をテキストプロンプト（"A cat"）で制御してきました。
しかし、「このポーズで」「この輪郭で」といった**空間的な制御**をテキストだけで指示するのは非常に困難です。

そこで、**画像条件（スケッチ、エッジ、骨格、深度）**を入力とする手法が登場しました。

### 🎨 画像間の変換（Image-to-Image Translation）

- **2017年 Pix2Pix (cGAN)**: 
  エッジ画像 → リアルな顔。U-Net Generator + パッチベースのDiscriminatorで実現。
- **2023年 ControlNet (Diffusion)**: 
  既存の強力なテキスト画像モデル（Stable Diffusionなど）を凍結し、輪郭などの空間条件を追加制御する画期的な手法。

### 📝 この章の構成

1. **空間条件の生成** — MNISTからエッジ画像を抽出
2. **Pix2Pixのアーキテクチャ** — 入力層への条件結合（コンカチ）
3. **簡易ControlNetの実装** — Zero-Convによるサイドブランチ
4. **学習と評価** — エッジを条件として内部構造を生成

In [ ]:
# ============================================================
# 環境設定
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from scipy.ndimage import gaussian_filter, map_coordinates
import warnings

warnings.filterwarnings('ignore')

def setup_japanese_font():
    japanese_fonts = ['Hiragino Sans', 'Yu Gothic', 'MS Gothic', 'Noto Sans CJK JP', 'IPAexGothic']
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 8)
print(f"Device: {device}")
print("✅ 環境設定完了")

---

## 1. 空間条件（エッジ）の抽出

MNISTの数字画像から、ControlNetの入力となる「条件画像」を作ります。
ここでは、Cannyエッジ風の輪郭と、「弾性変形」を加えたものを条件とします。

In [ ]:
# ============================================================
# 空間条件: 弾性変形エッジの生成
# ============================================================

def elastic_transform(image, alpha, sigma, random_state=None):
    """弾性変形で「手書きのゆらぎ」をシミュレート"""
    if random_state is None: random_state = np.random.RandomState(None)
    shape = image.shape
    dx = gaussian_filter((random_state.rand(*shape) * 2 - 1), sigma, mode="constant", cval=0) * alpha
    dy = gaussian_filter((random_state.rand(*shape) * 2 - 1), sigma, mode="constant", cval=0) * alpha
    x, y = np.meshgrid(np.arange(shape[1]), np.arange(shape[0]))
    indices = np.reshape(y+dy, (-1, 1)), np.reshape(x+dx, (-1, 1))
    return map_coordinates(image, indices, order=1, mode='reflect').reshape(shape)

def get_edge_condition(imgs_tensor):
    """バッチ画像からエッジ条件を計算（N, 1, 28, 28）"""
    conds = []
    for img in imgs_tensor:
        np_img = img.squeeze().numpy()
        np_img = elastic_transform(np_img, alpha=8, sigma=3)
        edges = np.abs(np.gradient(np_img)[0]) + np.abs(np.gradient(np_img)[1])
        conds.append((edges > edges.mean() * 0.5).astype(np.float32))
    return torch.tensor(np.array(conds)).unsqueeze(1)

# テスト表示
transform = transforms.ToTensor()
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)
sample_imgs, _ = next(iter(test_loader))
cond_imgs = get_edge_condition(sample_imgs)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(sample_imgs[i].squeeze(), cmap='gray')
    axes[0, i].set_title(f"元画像", fontsize=10)
    axes[1, i].imshow(cond_imgs[i].squeeze(), cmap='gray')
    axes[1, i].set_title(f"エッジ条件", fontsize=10)
    for row in range(2): axes[row, i].axis('off')

fig.suptitle('Pix2Pix / ControlNet の入力ペア
下: 入力条件 (エッジ)   上: ターゲット (実画像)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_315_01_edge_condition.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 2. Pix2Pix vs ControlNet の構造比較

どちらもU-Net型アーキテクチャですが、条件の入れ方が異なります。

### 🎨 Pix2Pix方式 (Concatenation)

入力画像のチャンネルに直接条件画像を結合（concat）します。
`(xₜ_ノイズ, 条件エッジ) -> [N, 2, 28, 28] -> 最初のConv層に入力`

- **欠点**: 最初のConv層の入力次元が変わるため、事前学習済みの重み（元のSDなど）を**完全に破壊**してしまい、ゼロから学習し直す必要があります。

### 🎨 ControlNet方式 (Zero-Conv サイドブランチ)

元のU-Netの重みを**完全に凍結（Lock）**し、同じ構造の「制御用コピー」を作ります。

```
  xₜ_ノイズ ───────────────(凍結U-Net)────────────────────→ 出力
     │                           ↑ (Zero-Convで合流)
     ├───> 制御モデル(Trainable) ──┘
     │
  条件画像 c
```

ZeroConvは「重みとバイアスが0のConv層」です。
学習初期は制御側の出力が**完全に0**になるため、凍結U-Netの事前学習の挙動を全く壊さずに学習をスタートできます。

In [ ]:
# ============================================================
# 簡易ControlNet搭載の拡散モデル
# ============================================================

class ZeroConv2d(nn.Conv2d):
    """重みとバイアスを0で初期化するConv2d層"""
    def __init__(self, in_c, out_c, kernel_size, padding=0):
        super().__init__(in_c, out_c, kernel_size, padding=padding)
        nn.init.constant_(self.weight, 0.0)
        nn.init.constant_(self.bias, 0.0)

class SimpleControlNet(nn.Module):
    """
    ControlNetの直感を捉え、事前学習(Linear層)に
    別のネットワーク(Conv層)から抽出した空間条件を注入する。
    MNIST用なのでU-NetではなくMLPベースの簡易版で概念を表現。
    """
    def __init__(self, dim=784, time_emb_dim=32):
        super().__init__()
        # 1. Base Model (本来はここに凍結されたSDが入る。今回は簡略化のため学習可能なMLP)
        self.time_mlp = nn.Sequential(nn.Linear(1, time_emb_dim), nn.SiLU(), nn.Linear(time_emb_dim, time_emb_dim))
        self.base_net = nn.Sequential(nn.Linear(dim + time_emb_dim, 512), nn.SiLU(), nn.Linear(512, 512), nn.SiLU())
        self.output_layer = nn.Linear(512, dim)

        # 2. ControlNet Branch (空間条件 c を処理)
        # 最初の層で c(1ch) と ノイズx(1ch) を受ける
        self.control_net = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1, stride=2), nn.ReLU(), # 14x14
            nn.Conv2d(32, 64, 3, padding=1, stride=2), nn.ReLU(), # 7x7
            nn.Flatten(),
            nn.Linear(64*7*7, 512)
        )

        # 3. Zero-Convolution (MLPの場合はZero-Linear)
        self.zero_linear = nn.Linear(512, 512)
        nn.init.constant_(self.zero_linear.weight, 0.0)
        nn.init.constant_(self.zero_linear.bias, 0.0)

    def forward(self, x_flat, t, c_img):
        # Base Model processing
        t_emb = self.time_mlp(t.float().unsqueeze(-1) / 100.0)
        base_h = self.base_net(torch.cat([x_flat, t_emb], dim=-1))

        # ControlNet processing (画像形式に戻す)
        x_img = x_flat.view(-1, 1, 28, 28)
        c_input = torch.cat([x_img, c_img], dim=1) # (N, 2, 28, 28)
        ctrl_h = self.control_net(c_input)

        # Zero-Conv による注入
        # 初期状態では ctrl_h * 0 = 0 なのでBaseの挙動を壊さない
        fused_h = base_h + self.zero_linear(ctrl_h)

        return self.output_layer(fused_h)

def get_schedule(n_steps=100, beta_start=1e-4, beta_end=0.02):
    betas = np.linspace(beta_start, beta_end, n_steps)
    alphas = 1.0 - betas
    return {'betas': torch.tensor(betas, dtype=torch.float32),
            'alphas': torch.tensor(alphas, dtype=torch.float32),
            'alpha_bars': torch.tensor(np.cumprod(alphas), dtype=torch.float32)}

schedule = get_schedule(100)
ab = schedule['alpha_bars'].to(device)

model = SimpleControlNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print("✅ Simple ControlNet アーキテクチャ構築完了")
print("   Base Modelの途中に Zero-Linear 層を通じて ControlNet側の特徴を注入します。")

In [ ]:
# ============================================================
# ControlNetの学習
# ============================================================

train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=True)

print("ControlNet学習中（空間エッジ条件）...")
for epoch in range(2):
    model.train()
    total = 0
    for x, _ in train_loader:
        x_img = x.to(device)                  # [N, 1, 28, 28], 値: [0, 1]
        x_flat = x_img.view(-1, 784) * 2 - 1  # [N, 784],     値: [-1, 1]

        # 空間エッジ条件を計算（N, 1, 28, 28）
        c_img = get_edge_condition(x.cpu()).to(device)

        t = torch.randint(0, 100, (x.shape[0],)).to(device)
        noise = torch.randn_like(x_flat)
        x_noisy = torch.sqrt(ab[t].unsqueeze(-1)) * x_flat + torch.sqrt(1 - ab[t].unsqueeze(-1)) * noise

        pred_noise = model(x_noisy, t, c_img)
        loss = nn.functional.mse_loss(pred_noise, noise)

        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1}/15 | Loss: {total/len(train_loader):.6f}")

model.eval()
print("✅ 学習完了")

---

## 3. 実験と評価: 境界線から数字を生成

In [ ]:
# ============================================================
# エッジ条件を与えて画像をSampling（DDIM）
# ============================================================

@torch.no_grad()
def sample_controlnet(model, c_imgs, n_steps=5):
    step_idx = np.linspace(0, 99, n_steps + 1, dtype=int)
    x = torch.randn(c_imgs.shape[0], 784).to(device)
    for i in range(n_steps, 0, -1):
        t_c, t_p = step_idx[i], step_idx[i - 1]
        t_t = torch.full((x.shape[0],), t_c, dtype=torch.long).to(device)
        eps = model(x, t_t, c_imgs)
        x0_p = (x - torch.sqrt(1 - ab[t_c]) * eps) / torch.sqrt(ab[t_c])
        x = torch.sqrt(ab[t_p]) * x0_p + torch.sqrt(1 - ab[t_p]) * eps
    return x

# テスト: エッジ条件を与えて生成
test_imgs, _ = next(iter(test_loader))
c_imgs = get_edge_condition(test_imgs[:10]).to(device)

print("Sampling中...")
x_gen = sample_controlnet(model, c_imgs, n_steps=5)

fig, axes = plt.subplots(3, 10, figsize=(16, 5))
for i in range(10):
    axes[0, i].imshow(test_imgs[i].squeeze().numpy(), cmap='gray')
    axes[1, i].imshow(c_imgs[i].squeeze().cpu().numpy(), cmap='gray')
    img_g = ((x_gen[i].cpu().numpy().reshape(28, 28) + 1) / 2).clip(0, 1)
    axes[2, i].imshow(img_g, cmap='gray')
    for r in range(3): axes[r, i].axis('off')

axes[0, 0].set_ylabel('元(参照)', fontsize=11, rotation=0, labelpad=40)
axes[1, 0].set_ylabel('エッジ条件', fontsize=11, rotation=0, labelpad=40)
axes[2, 0].set_ylabel('ControlNet
生成結果', fontsize=11, rotation=0, labelpad=40)

fig.suptitle('ControlNet: 空間的な輪郭（エッジ）条件から画像を生成
エッジの形状に完璧に従った画像が生成される',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_315_02_controlnet_generation.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 ControlNetは空間的構造（どこに線を引くか）を完全に支配します。")
print("   SDでは「猫」というテキストとエッジ条件を与えると、指定した同じ姿勢の「猫」が生成されます。")

---

## まとめ

### 🎯 学んだこと

- ✓ **空間的条件の注入**: Pix2Pixは入力にconcatするのに対し、ControlNetはサイドブランチを使う
- ✓ **Zero-Convの役割**: 重みを0で初期化することで、事前学習済みモデルの振る舞いを壊さずにファインチューニングを開始できる
- ✓ **Image-to-Image**: エッジ、深度、ポーズなどの空間情報を条件として、生成画像のレイアウトを直接制御できる

### ✅ 学習チェックリスト

- [ ] ControlNetが事前学習モデルの重みを凍結する理由を説明できるか？
- [ ] Zero-Convの初期出力が0になる計算メカニズムを説明できるか？

---

**次のステップ**: Unit 14 最終章！ ノートブック316「画像変容ユニット 総合演習」で、これまで学んだ技術を組み合わせた実装に挑戦します。